# Adult Dataset Preprocessing

This notebook reproduces the preprocessing protocol used for the **Adult** dataset in the GradTree paper:

- Use the 32,561-row `adult.data` file only.
- Treat categorical missing values as an explicit `Unknown` category.
- Create a random 80/20 train-test split, then an 80/20 train-validation split.
- Apply leave-one-out target encoding to all categorical features.
- Apply a normal quantile transform to all features.
- Apply SMOTE to the training split only because the minority class is below 25%.

The notebook can be run directly in Google Colab. Generated datasets and fitted transformers are written under `preprocess/adult_output/` and are intentionally excluded from Git.

## 1. Install dependencies

Colab already includes pandas, NumPy, and scikit-learn. The two packages below are needed for target encoding and SMOTE.

In [ ]:
%pip install -q category_encoders imbalanced-learn

In [ ]:
from __future__ import annotations

import json
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from category_encoders import LeaveOneOutEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer

SEED = 42
QUANTILE_NOISE = 1e-3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "preprocess":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset" / "adult"
OUTPUT_DIR = PROJECT_ROOT / "preprocess" / "adult_output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Download the source data

Only `adult.data` is used. It contains the 32,561 examples reported by the paper. The original UCI URL is tried first, with the GradTree authors' mirror as a fallback.

In [ ]:
ADULT_URLS = [
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "https://raw.githubusercontent.com/s-marton/GradTree/master/experiments_paper_gradtree/real_world_datasets/Adult/adult.data",
]
ADULT_PATH = DATA_DIR / "adult.data"

if not ADULT_PATH.exists() or ADULT_PATH.stat().st_size == 0:
    errors = []
    for url in ADULT_URLS:
        try:
            urllib.request.urlretrieve(url, ADULT_PATH)
            break
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    else:
        raise RuntimeError("Could not download adult.data:\n" + "\n".join(errors))

print(f"Source file: {ADULT_PATH}")
print(f"Size: {ADULT_PATH.stat().st_size:,} bytes")

## 3. Load and clean the data

The paper keeps all 14 features. The `?` values in `workclass`, `occupation`, and `native-country` are retained as an explicit `Unknown` category because the paper does not specify an imputation rule.

In [ ]:
COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income",
]
FEATURES = COLUMNS[:-1]
CATEGORICAL_FEATURES = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]
NUMERIC_FEATURES = [column for column in FEATURES if column not in CATEGORICAL_FEATURES]

data = pd.read_csv(
    ADULT_PATH,
    names=COLUMNS,
    skipinitialspace=True,
    na_values=["?"],
    skip_blank_lines=True,
)

for column in data.select_dtypes(include=["object", "string"]).columns:
    data[column] = data[column].str.strip()

missing_counts = {column: int(count) for column, count in data.isna().sum().items()}
for column in CATEGORICAL_FEATURES:
    data[column] = data[column].fillna("Unknown")
data = data.dropna(subset=["income"]).reset_index(drop=True)

assert len(data) == 32_561, f"Expected 32,561 rows, found {len(data):,}"
assert len(FEATURES) == 14
assert not data[FEATURES].isna().any().any()

y = data.pop("income").eq(">50K").astype("int8")
X = data[FEATURES].copy()

target_counts = {int(label): int(count) for label, count in y.value_counts().sort_index().items()}
assert target_counts == {0: 24_720, 1: 7_841}

print(f"Rows: {len(X):,}")
print(f"Features: {X.shape[1]}")
print(f"Target counts (0=<=50K, 1=>50K): {target_counts}")
print(f"Missing values by column before unknown-category filling: {missing_counts}")
display(data.head())

## 4. Create train, validation, and test splits

This follows the paper literally: a random 80/20 train-test split followed by a random 80/20 split of the training portion. Splitting happens before fitting the target encoder or quantile transformer to prevent leakage.

In [ ]:
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=0.20,
    random_state=SEED,
)

splits = {
    "train": (X_train, y_train),
    "valid": (X_valid, y_valid),
    "test": (X_test, y_test),
}
for name, (split_X, split_y) in splits.items():
    positive_ratio = float(split_y.mean())
    print(f"{name:5s}: rows={len(split_X):,}, features={split_X.shape[1]}, positive_ratio={positive_ratio:.4f}")

## 5. Leave-one-out target encoding

For each training category, its encoded value is the target mean after excluding the current row. Validation and test categories use the full training target mean. Unknown categories fall back to the global training target mean.

In [ ]:
encoder = LeaveOneOutEncoder(
    cols=CATEGORICAL_FEATURES,
    handle_missing="value",
    handle_unknown="value",
    random_state=SEED,
)

X_train_encoded = encoder.fit_transform(X_train, y_train)
X_valid_encoded = encoder.transform(X_valid)
X_test_encoded = encoder.transform(X_test)

feature_names = list(X_train_encoded.columns)
assert len(feature_names) == 14
assert np.isfinite(X_train_encoded.to_numpy()).all()
assert np.isfinite(X_valid_encoded.to_numpy()).all()
assert np.isfinite(X_test_encoded.to_numpy()).all()

print(f"Encoded feature count: {len(feature_names)}")
display(X_train_encoded.head())

## 6. Quantile-normal transformation

A single quantile transformer is fitted to all 14 training features and then reused for validation and test data. Following the authors' implementation, a small seeded Gaussian jitter (`1e-3`) is added only to the copy used for fitting so tied categorical encodings do not create degenerate quantiles.

In [ ]:
quantile_fit_data = X_train_encoded.to_numpy(dtype=np.float64, copy=True)
rng = np.random.RandomState(SEED)
stds = quantile_fit_data.std(axis=0, keepdims=True)
noise_std = QUANTILE_NOISE / np.maximum(stds, QUANTILE_NOISE)
quantile_fit_data += noise_std * rng.randn(*quantile_fit_data.shape)

quantile_transformer = QuantileTransformer(
    n_quantiles=min(1_000, len(X_train_encoded)),
    output_distribution="normal",
    subsample=None,
    random_state=SEED,
)
quantile_transformer.fit(quantile_fit_data)

def apply_quantile_transform(frame: pd.DataFrame) -> pd.DataFrame:
    values = quantile_transformer.transform(frame.to_numpy(dtype=np.float64))
    return pd.DataFrame(values, columns=feature_names, index=frame.index)

X_train_transformed = apply_quantile_transform(X_train_encoded)
X_valid_transformed = apply_quantile_transform(X_valid_encoded)
X_test_transformed = apply_quantile_transform(X_test_encoded)

transformed_stats = X_train_transformed.agg(["mean", "std"]).T
assert X_train_transformed.shape == (len(X_train), 14)
assert np.isfinite(X_train_transformed.to_numpy()).all()
display(transformed_stats)

## 7. Rebalance the training data

For binary classification, the paper applies SMOTE when the minority fraction is at most `25 / (2 - 1) = 25%`. Adult's minority class is approximately 24.1%, so SMOTE is applied. Validation and test data are never resampled.

In [ ]:
num_classes = int(y_train.nunique())
minority_ratio = float(y_train.value_counts(normalize=True).min())
balance_threshold = 0.25 / (num_classes - 1)
smote_applied = minority_ratio <= balance_threshold

if smote_applied:
    X_train_final, y_train_final = SMOTE(random_state=SEED).fit_resample(
        X_train_transformed,
        y_train,
    )
    X_train_final = pd.DataFrame(X_train_final, columns=feature_names)
    y_train_final = pd.Series(y_train_final, name="income", dtype="int8")
else:
    X_train_final = X_train_transformed.copy()
    y_train_final = y_train.copy()

assert len(X_valid_transformed) == len(y_valid)
assert len(X_test_transformed) == len(y_test)
assert np.isfinite(X_train_final.to_numpy()).all()

print(f"Minority ratio before balancing: {minority_ratio:.4f}")
print(f"SMOTE threshold: {balance_threshold:.2f}")
print(f"SMOTE applied: {smote_applied}")
print(f"Training rows after balancing: {len(X_train_final):,}")
print(f"Training target counts after balancing: {y_train_final.value_counts().to_dict()}")

## 8. Save processed datasets and fitted transformers

The saved encoder and quantile transformer must be reused if additional test data are processed later. Generated files are ignored by Git.

In [ ]:
preprocessor_bundle = {
    "encoder": encoder,
    "quantile_transformer": quantile_transformer,
    "feature_names": feature_names,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "random_seed": SEED,
    "quantile_noise": QUANTILE_NOISE,
}
joblib.dump(preprocessor_bundle, OUTPUT_DIR / "adult_preprocessor.joblib")

def save_split(name: str, features: pd.DataFrame, target: pd.Series) -> None:
    output = features.copy()
    output["income"] = np.asarray(target, dtype=np.int8)
    output.to_csv(OUTPUT_DIR / f"adult_{name}.csv", index=False)

save_split("train", X_train_final, y_train_final)
save_split("valid", X_valid_transformed, y_valid)
save_split("test", X_test_transformed, y_test)

summary = {
    "source_rows": int(len(X)),
    "feature_count": int(len(FEATURES)),
    "original_target_counts": target_counts,
    "missing_counts_before_unknown_category": missing_counts,
    "split_rows": {name: int(len(split_X)) for name, (split_X, _) in splits.items()},
    "train_rows_after_smote": int(len(X_train_final)),
    "minority_ratio_before_smote": minority_ratio,
    "smote_threshold": balance_threshold,
    "smote_applied": bool(smote_applied),
    "train_target_counts_after_smote": {
        int(label): int(count) for label, count in y_train_final.value_counts().sort_index().items()
    },
}
(OUTPUT_DIR / "adult_preprocessing_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("Saved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")